# SETUP FOR BETTER GPU

1) Install UCSD VPN: https://blink.ucsd.edu/technology/network/connections/off-campus/VPN/
2) ssh on local machine to ssh username@dsmlp-login.ucsd.edu where username is <username>@ucsd.edu
3) Run the following command on the ssh: launch.sh -c 8 -m 16 -g 1 -v a30
4) Open the link it provides
5) Make sure to **shut down** the pod when you're done from File > Shut Down. And then exit ssh from your terminal

6) If it says GPU already in use (1 of 1 allocated): run the following "kubectl get pods" and "kubectl delete pod <paste-pod-name-here>"


## Starter Code for Datahub

### IF THIS IS YOUR FIRST TIME RUNNING THIS ON DATA HUB
You need to run the cell immediately below so that you can install/setup the kernel. Its called "Python (cse151b)". You should **always** select this kernel in the top right of jupyter. 

### If you're coming back, make sure you re-run everything from the next cell (imports) and further down

The purpose of this starter code is so that you can basically duplicate this and run your experiments in a separate notebook.

# START HERE IF ITS YOUR FIRST TIME HERE

In [ ]:
# =========================================================
# DATAHUB SETUP & IMPORTS. (May take a while!) Comment me out after first run!
# =========================================================

# remove old kernel
!jupyter kernelspec uninstall -y cse151b

# Remove old .venv
!rm -rf .venv

# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

print("Old environment and kernel destroyed.")

# Create a virtual environment
!~/.local/bin/uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers vllm==0.19.1 tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter scikit-learn pandas

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding. Then click on current kernel -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'. Restart kernel again. Comment out this code!")

# START HERE IF YOU'VE ALREADY SETUP! 

# MAKE SURE TO RESTART KERNEL BEFORE CONTINUING!

## Library Setup

In [ ]:
!source ./.venv/bin/activate

import json
import os
import random

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

import re
import sys
from pathlib import Path
from typing import Optional

import pandas as pd
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm
from datetime import datetime 
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

## Config + Data Loading
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0" 
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/datahub_experiments.jsonl"
MAX_TOKENS  = 16384

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

data = [json.loads(line) for line in open(DATA_PATH)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

# Preview one MCQ and one free-form item
mcq_sample  = next(d for d in data if d.get("options"))
free_sample = next(d for d in data if not d.get("options"))

print("\n── MCQ sample ──")
print(json.dumps(mcq_sample, indent=2))
print("\n── Free-form sample ──")
print(json.dumps(free_sample, indent=2))

## Hyperparam Sandbox + LLM Setup

If this doesn't work, what worked for me was:
1) shut down ALL OTHER kernels in the directory
2) re-run the first cell (commented out one)

If it still fails:
1) run "pkill -9 -f python" in terminal. This will kill the datahub pod and (most likely) wipe /dev/shm
2) re-launch the container
3) ensure all other kernels are shut down and go from the top.

In [ ]:
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# tokenizer.pad_token = tokenizer.eos_token

# llm = LLM(
#     model=MODEL_ID,
#     quantization="bitsandbytes",
#     load_format="bitsandbytes",
#     enable_prefix_caching=False,
#     enforce_eager=True,
#     gpu_memory_utilization=0.85,
#     max_model_len=16384,
#     trust_remote_code=True,
#     max_num_seqs=256,
#     max_num_batched_tokens=32768,
# )

# sampling_params = SamplingParams(
#     max_tokens=MAX_TOKENS,
#     temperature=0.6,
#     top_p=0.95,
#     top_k=20,
#     min_p=0.0,
#     presence_penalty=0.0,
#     repetition_penalty=1.0,
# )

# print("Model and parameters loaded successfully in Eager Mode.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16", 
    quantization=None, 
    load_format="auto",
    enable_prefix_caching=True, 
    enforce_eager=False,        
    gpu_memory_utilization=0.95, 
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    presence_penalty=0.0,
    repetition_penalty=1.05, # slight penalty for repetition
)

print("Model loaded with CUDAGraph + BF16.")

## Experiment/Submission Config - CONFIGURE ME!

In [ ]:
IS_EXPERIMENT   = True               # Set to False to run the full 1126 dataset
EXPERIMENT_NAME = "no_few_shot_baseline"           # Only used if IS_EXPERIMENT is True
SAVE_EVAL       = True               # Set to False for private test set (no ground truth)

# Dynamic Naming Logic
date_str = datetime.now().strftime("%Y-%m-%d")

if IS_EXPERIMENT:
    output_filename = f"experiment_{EXPERIMENT_NAME}_{date_str}.jsonl"
else:
    output_filename = f"submission_{date_str}.jsonl"

OUTPUT_PATH = Path("results") / output_filename
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"RUN MODE : {'EXPERIMENT (50 samples)' if IS_EXPERIMENT else 'FULL DATASET (1126 items)'}")
print(f"SAVE TO  : {OUTPUT_PATH}")

## Prompt Engineering Sandbox

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

## Dataset Generation

In [ ]:
if IS_EXPERIMENT:
    SUBSET_SIZE = 50
    RANDOM_SEED = 42
    random.seed(RANDOM_SEED)

    mcq_pool  = [d for d in data if d.get("options")]
    free_pool = [d for d in data if not d.get("options")]

    mcq_ratio = len(mcq_pool) / len(data)
    n_mcq  = max(1, round(SUBSET_SIZE * mcq_ratio))
    n_free = SUBSET_SIZE - n_mcq

    data_to_run = random.sample(mcq_pool, n_mcq) + random.sample(free_pool, n_free)
    random.shuffle(data_to_run)
else:
    data_to_run = data

print(f"Data ready. Queuing {len(data_to_run)} questions for inference.")

## Inference 

In [ ]:
prompts = []
for item in data_to_run:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

print(f"Generating responses for {len(prompts)} questions...")

# Chunking to prevent EngineDead errors
BATCH_SIZE = 8   
all_outputs = []

for i in range(0, len(prompts), BATCH_SIZE):
    batch = prompts[i:i + BATCH_SIZE]
    print(f"Processing batch {i//BATCH_SIZE + 1} / {(len(prompts) + BATCH_SIZE - 1) // BATCH_SIZE}")
    outputs = llm.generate(batch, sampling_params=sampling_params)
    all_outputs.extend(outputs)

responses = [out.outputs[0].text.strip() for out in all_outputs]

# Preview first 3
for i in range(min(3, len(responses))):
    print(f"\n── Response {i} (id={data_to_run[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

## Score Responses + Summary

Scoring differs by question type:

- **MCQ**: extract the predicted letter from `\boxed{}` and compare to the gold letter (exact match).
- **Free-form**: use `Judger.auto_judge()` which handles symbolic and numeric equivalence.

Each result record contains `{id, is_mcq, gold, response, correct}`.

In [ ]:
# Load Judger
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m: return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

results = []
for item, response in tqdm(zip(data_to_run, responses), total=len(data_to_run), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = extract_letter(response) == str(gold).strip().upper()
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
        except Exception:
            correct = False

    results.append({
        "id": item.get("id"), "is_mcq": is_mcq, "gold": gold,
        "response": response, "correct": correct,
    })

# --- PANDAS METRICS TABLE ---
def generate_metrics_df(results):
    rows = []
    for subset_name, is_mcq_flag in [("Overall", None), ("MCQ", True), ("Free-Form", False)]:
        if is_mcq_flag is None:
            subset = results
        else:
            subset = [r for r in results if r["is_mcq"] == is_mcq_flag]
            
        if not subset: continue
            
        y_true = [1] * len(subset)
        y_pred = [1 if r["correct"] else 0 for r in subset]
        
        rows.append({
            "Category": subset_name,
            "Accuracy": f"{accuracy_score(y_true, y_pred):.2%}",
            "F1 Score": round(f1_score(y_true, y_pred, zero_division=0), 3),
            "Correct": sum(y_pred),
            "Total": len(y_pred)
        })
    return pd.DataFrame(rows).set_index("Category")

metrics_df = generate_metrics_df(results)
display(metrics_df)

## Save Results

Results are written as newline-delimited JSON.

**With evaluation** (public set — you have ground-truth):  
Each line: `{id, is_mcq, gold, response, correct}`

**Without evaluation** (private test set — no ground-truth available):  
Each line: `{id, is_mcq, response}` — omit `gold` and `correct`.

Toggle `SAVE_EVAL` below accordingly.

In [ ]:
with open(OUTPUT_PATH, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {OUTPUT_PATH}")